# Static Time-Series Chart

### Annual Drug Offense Reports, 2003–2025


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams["font.family"] = "DejaVu Serif"

In [ ]:
df = pd.read_csv("../SF_Crime_Merged_Focus_2003_to_2025.csv")
df["Incident Date"] = pd.to_datetime(df["Incident Date"])

drug = df[df["Focus Crime"] == "Drug Offense"].copy()
drug["Year"] = drug["Incident Date"].dt.year
drug = drug[drug["Year"] < 2026]   # exclude partial 2026 data

yearly = drug.groupby("Year").size().reset_index(name="Count")

In [ ]:
# ── Colour palette ──────────────────────────────────────────────────────────
C_LINE  = "#1a3a5c"   # dark navy for the main line
C_ERA1  = "#d6e4f0"   # light blue  – peak era shading
C_ERA2  = "#fde8d8"   # light amber – COVID trough shading
C_ERA3  = "#d5f0e0"   # light green – rebound era shading
C_ANNOT = "#c0392b"   # red for annotation markers
C_GRID  = "#e8e8e8"

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

years  = yearly["Year"].values
counts = yearly["Count"].values

# ── Era shading ──────────────────────────────────────────────────────────────
ax.axvspan(2003,   2009.5, color=C_ERA1,   alpha=0.55, zorder=0)   # peak era
ax.axvspan(2009.5, 2021.5, color="#eef5fb", alpha=0.55, zorder=0)  # long decline
ax.axvspan(2019.5, 2021.5, color=C_ERA2,   alpha=0.65, zorder=0)   # COVID trough
ax.axvspan(2021.5, 2025,   color=C_ERA3,   alpha=0.55, zorder=0)   # rebound

# ── Main line ────────────────────────────────────────────────────────────────
ax.plot(years, counts, color=C_LINE, linewidth=2.5, zorder=3)
ax.fill_between(years, counts, alpha=0.08, color=C_LINE, zorder=2)

# ── Key-point markers ────────────────────────────────────────────────────────
key_points = {
    2008: (8405, "Peak: 8,405", "above"),
    2021: ( 942, "Trough: 942", "below"),
    2025: (4451, "2025: 4,451", "above"),
}
for yr, (cnt, label, pos) in key_points.items():
    ax.scatter(yr, cnt, color=C_ANNOT, s=60, zorder=5)
    offset = 280 if pos == "above" else -450
    ax.annotate(
        label,
        xy=(yr, cnt), xytext=(yr, cnt + offset),
        fontsize=9.5, color=C_ANNOT, fontweight="bold", ha="center",
        arrowprops=dict(arrowstyle="-", color=C_ANNOT, lw=1.2),
    )

# ── Policy / event callouts ──────────────────────────────────────────────────
prop47_cnt = int(yearly[yearly["Year"] == 2015]["Count"].values[0])
ax.annotate(
    "Prop 47\n(Nov 2014)",
    xy=(2015, prop47_cnt), xytext=(2015.8, 3200),
    fontsize=8.5, color="#555555", ha="left",
    arrowprops=dict(arrowstyle="->", color="#888888", lw=1.0),
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#cccccc", lw=0.8),
)

covid_cnt = int(yearly[yearly["Year"] == 2020]["Count"].values[0])
ax.annotate(
    "COVID-19\nlockdowns",
    xy=(2020, covid_cnt), xytext=(2017.5, 1900),
    fontsize=8.5, color="#555555", ha="left",
    arrowprops=dict(arrowstyle="->", color="#888888", lw=1.0),
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#cccccc", lw=0.8),
)

# ── Era labels ───────────────────────────────────────────────────────────────
ax.text(2005.0, 8750, "Peak era",     fontsize=8, color="#3a7abf", style="italic")
ax.text(2013.0, 8750, "Long decline", fontsize=8, color="#3a7abf", style="italic")
ax.text(2022.3, 8750, "Rebound",      fontsize=8, color="#2a8a4a", style="italic")

# ── Axes / grid ──────────────────────────────────────────────────────────────
ax.set_xlim(2002.5, 2025.5)
ax.set_ylim(0, 9500)
ax.set_xticks(range(2003, 2026, 2))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.grid(axis="y", color=C_GRID, linewidth=0.8, zorder=1)
ax.tick_params(axis="both", labelsize=9.5)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color("#cccccc")
ax.spines["bottom"].set_color("#cccccc")

# ── Title & axis labels ──────────────────────────────────────────────────────
ax.set_title(
    "Drug Offense Reports in San Francisco, 2003–2025",
    fontsize=15, fontweight="bold", pad=14, loc="left",
)
ax.set_xlabel("Year", fontsize=10, labelpad=6)
ax.set_ylabel("Annual incident reports", fontsize=10, labelpad=8)

# ── Legend ───────────────────────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(color=C_ERA1,    alpha=0.55, label="Peak era (2003–2009)"),
    mpatches.Patch(color="#eef5fb", alpha=0.55, label="Long decline (2010–2021)"),
    mpatches.Patch(color=C_ERA2,    alpha=0.65, label="COVID trough (2020–2021)"),
    mpatches.Patch(color=C_ERA3,    alpha=0.55, label="Rebound (2022–2025)"),
]
ax.legend(handles=legend_patches, loc="upper right",
          fontsize=8.5, framealpha=0.9, edgecolor="#cccccc")

plt.tight_layout()
plt.savefig("drug_offense_timeseries.png", dpi=150, bbox_inches="tight")
plt.savefig("drug_offense_timeseries.svg", bbox_inches="tight")
plt.show()
print("Saved: drug_offense_timeseries.png / .svg")